# Model Evaluation Notebook

I use this notebook to pull the project's main evaluation work into one place. It covers the tabular transaction path, the anomaly model, the NLP artifact, and the CV artifact so the report and presentation can point to one reproducible workflow.


In [ ]:
from pathlib import Path
import json

import joblib
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.calibration import calibration_curve
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score, roc_curve, precision_recall_curve
from sklearn.model_selection import train_test_split

from src.train.evaluate_anomaly_model import evaluate_anomaly_model
from src.train.evaluate_cv_model import evaluate_cv_model
from src.train.model_paths import NLP_MODEL

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / 'data'
DOCS_DIR = PROJECT_ROOT / 'docs'
FIGURES_DIR = DOCS_DIR / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


## Tabular Transaction Evaluation

I train a compact Random Forest here because I want a repeatable baseline for ROC-AUC, PR-AUC, calibration, and feature importance using the prepared transaction dataset.


In [ ]:
tabular_df = pd.read_csv(DATA_DIR / 'processed' / 'transactions' / 'clean_validation.csv')
X = tabular_df.drop(columns=['is_fraud'])
y = tabular_df['is_fraud'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
tabular_probs = rf.predict_proba(X_test)[:, 1]

tabular_metrics = {
    'roc_auc': float(roc_auc_score(y_test, tabular_probs)),
    'pr_auc': float(average_precision_score(y_test, tabular_probs)),
}
tabular_metrics


In [ ]:
fpr, tpr, _ = roc_curve(y_test, tabular_probs)
precision, recall, _ = precision_recall_curve(y_test, tabular_probs)
prob_true, prob_pred = calibration_curve(y_test, tabular_probs, n_bins=10)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(fpr, tpr, label=f"ROC AUC = {tabular_metrics['roc_auc']:.3f}")
axes[0].plot([0, 1], [0, 1], linestyle='--', color='gray')
axes[0].set_title('ROC Curve')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].legend()

axes[1].plot(recall, precision, label=f"PR AUC = {tabular_metrics['pr_auc']:.3f}")
axes[1].set_title('Precision-Recall Curve')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].legend()

axes[2].plot(prob_pred, prob_true, marker='o')
axes[2].plot([0, 1], [0, 1], linestyle='--', color='gray')
axes[2].set_title('Calibration Curve')
axes[2].set_xlabel('Mean Predicted Probability')
axes[2].set_ylabel('Fraction of Positives')

fig.tight_layout()
fig.savefig(FIGURES_DIR / 'tabular_curves.png', dpi=160)
plt.show()


In [ ]:
feature_importance = (
    pd.Series(rf.feature_importances_, index=X.columns)
    .sort_values(ascending=False)
    .head(12)
)

ax = feature_importance.sort_values().plot(kind='barh', figsize=(8, 5), title='Top Tabular Feature Importances')
ax.figure.tight_layout()
ax.figure.savefig(FIGURES_DIR / 'tabular_feature_importance.png', dpi=160)
feature_importance


## Anomaly and CV Evaluation

I reuse the dedicated evaluation helpers here so the notebook stays aligned with the scripts that support the issue work.


In [ ]:
anomaly_summary = evaluate_anomaly_model()
cv_summary = evaluate_cv_model()
anomaly_summary, cv_summary


## NLP Artifact Check

I keep the NLP step lighter here because the main repo already stores the trained SMS artifact. This cell confirms whether that artifact exists and can be referenced in the report.


In [ ]:
nlp_exists = NLP_MODEL.exists()
nlp_exists


In [ ]:
summary = {
    'tabular_metrics': tabular_metrics,
    'anomaly_summary': anomaly_summary,
    'cv_summary': cv_summary,
    'nlp_artifact_available': bool(nlp_exists),
}
output_path = DOCS_DIR / 'model_evaluation_summary.json'
output_path.write_text(json.dumps(summary, indent=2))
summary
